In [1]:
import sqlite3
import sys
import pandas as pd
from pathlib import Path
import os
from IPython.display import display, Markdown

notebook_path = Path(os.getcwd())
root_dir = notebook_path
while root_dir.name != "data_warehouse" and root_dir.parent != root_dir:
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))    

from utils.hyperlocal_macro_factory import MacroFeatureEngineV2
engine = MacroFeatureEngineV2(geography_daemon=None)


⚙️ Engine initialized cleanly. Detected right-censored IRS ceiling as tax year: 2022


In [2]:
import pandas as pd
from utils.hyperlocal_macro_factory import MacroFeatureEngineV2

engine = MacroFeatureEngineV2(geography_daemon=None)

state_only_metrics = engine.extract_comprehensive_macro_profile(county_fips="01000", loan_vintage_year=2024)
county_metrics = engine.extract_comprehensive_macro_profile(county_fips="48085", loan_vintage_year=2024)
industry_metrics = engine.extract_comprehensive_macro_profile(county_fips="48085", naics_4d="7225",loan_vintage_year=2024)

df_grid = pd.DataFrame([state_only_metrics, county_metrics, industry_metrics]).T
df_grid.columns = ["Alabama State Core (01000)", "Collin County Metro (48085)", "Restaurants in Collin County"]
df_grid


⚙️ Engine initialized cleanly. Detected right-censored IRS ceiling as tax year: 2022


,Alabama State Core (01000),Collin County Metro (48085),Restaurants in Collin County
target_fips,01000,48085,48085
target_year,2024,2024,2024
target_naics,GENERAL,GENERAL,7225
spatial_governance_flag,STATE_LEVEL_CORE,TRUE_METRO,TRUE_METRO
macro_wealth_cushion,NaN,0.049359,0.049359
filer_density_velocity,NaN,0.148082,0.148082
household_dependency_ratio,NaN,1.935865,1.935865
labor_pool_structural_friction,NaN,1.255172,1.255172
wage_diversification_index,NaN,0.982583,0.982583
wage_to_filer_disconnect_index,NaN,-0.00822,-0.00822


In [3]:
def generate_markdown_loan_dashboard(profile_data: dict) -> str:
    """
    Transforms the raw MacroFeatureEngineV2 dictionary output into a 
    standardized institutional underwriting presentation layout.
    """
    fips = profile_data.get("target_fips", "UNKNOWN")
    year = profile_data.get("target_year", "UNKNOWN")
    naics = profile_data.get("target_naics", "GENERAL")
    flag = profile_data.get("spatial_governance_flag", "UNKNOWN")
    
    # Extract raw metrics with fallback protections
    m_cushion = profile_data.get("macro_wealth_cushion", 0.0) or 0.0
    f_velocity = profile_data.get("filer_density_velocity", 0.0) or 0.0
    h_dependency = profile_data.get("household_dependency_ratio", 0.0) or 0.0
    l_friction = profile_data.get("labor_pool_structural_friction", 0.0) or 0.0
    w_diversification = profile_data.get("wage_diversification_index", 1.0) or 1.0
    w_disconnect = profile_data.get("wage_to_filer_disconnect_index", 0.0) or 0.0
    s_momentum = profile_data.get("state_coincident_momentum", 0.0) or 0.0
    y_spread = profile_data.get("sovereign_yield_spread", 0.25) or 0.25
    c_sentiment = profile_data.get("macro_consumer_sentiment", 70.0) or 70.0
    p_turnover = profile_data.get("state_private_turnover_rate", 0.0) or 0.0
    n_job_flow = profile_data.get("state_net_job_flow_count", 0) or 0
    m_saturation = profile_data.get("industry_market_saturation_lq", None)

    md = f"""# 🛸 ENTERPRISE UNDERWRITING RISK REPORT: MACRO-METRIC ENGINE
---
### 📅 COHORT TARGET: FIPS [{fips}] | VINTAGE YEAR [{year}] | NAICS [{naics}]
**Spatial Governance Execution Flag:** `{flag}`

---

## 💎 DIMENSION 1: PASSIVE WEALTH DEPTH & MIGRATION PROFILES (IRS SOI)

### 📈 Macro Wealth Cushion
* **Current Cohort Result:** `{m_cushion:.4f}`
* **Spectrum of Expectations:**
  * **High (≥ 0.120):** Strong local liquidity cushion. High concentration of investment dividends and interest relative to labor wages, indicating local wealth insulation.
  * **Average (0.040 to 0.119):** Standard balanced marketplace. Consumer spending is supported primarily by active jobs, with normal household savings reserves.
  * **Low (< 0.040):** High-risk capital constraints. The community depends entirely on immediate payroll cycles; households lack passive liquidity cushions during stress.

### 🚀 Filer Density Velocity
* **Current Cohort Result:** `{f_velocity:+.4f} ({f_velocity*100:+.2f}%)`
* **Spectrum of Expectations:**
  * **High (≥ +0.050):** Rapid regional expansion. Net taxpayer population grew by 5%+ over the rolling 5-year window, indicating strong business and home demand.
  * **Average (-0.020 to +0.049):** Stable economic baseline. Normal household turnover and steady organic migration patterns.
  * **Low (< -0.020):** Regional contraction / Capital flight. Taxpayers are leaving the market, threatening the local tax base and commercial asset values.

### 🪆 Household Dependency Ratio
* **Current Cohort Result:** `{h_dependency:.4f}`
* **Spectrum of Expectations:**
  * **High (≥ 2.20):** Demographically vulnerable. Large family sizes or a high non-working dependent population relative to tax filers, tightening disposable household income.
  * **Average (1.60 to 2.19):** Balanced family demographic footprint, matching national baseline cost structures.
  * **Low (< 1.60):** Favorable household cash flow profile. High concentration of single filers or dual-income households with low dependency constraints.

---

## 🛠️ DIMENSION 2: LABOR MARKET DYNAMICS & FRICTION (BLS LAUS / CENSUS QWI)

### ⛓️ Labor Pool Structural Friction (Workforce Coefficient of Variation)
* **Current Cohort Result:** `{l_friction:.4f}`
* **Spectrum of Expectations:**
  * **High (≥ 0.060):** Volatile workforce environment. Erratic labor force counts or extreme seasonal employment spikes that can interrupt corporate operations.
  * **Average (0.015 to 0.059):** Stable labor pool layout. Predictable worker availability for consistent regional business hiring.
  * **Low (< 0.015):** Flat workforce landscape. Indicates a stagnant or rigid local labor market, making it difficult for scaling firms to find and hire new staff.

"""

    if m_saturation is not None:
        md += f"""### 🎯 Industry Market Saturation (Location Quotient)
* **Current Cohort Result:** `{m_saturation:.4f}`
* **Spectrum of Expectations:**
  * **High (≥ 1.25):** Highly specialized exporter sector. The market has an intense cluster of this business type compared to the US baseline, making it sensitive to sector cycles.
  * **Average (0.75 to 1.24):** Equilibrium cluster density. Local business volume matches standard domestic consumption patterns perfectly.
  * **Low (< 0.75):** Underserved local market. The sector is underrepresented, signaling potential market entry space or weak regional consumer demand.

"""

    md += f"""---

## 🔀 DIMENSION 3: ADVANCED REGIONAL STRUCTURE & DISCONNECT SPREADS (BLS QCEW)

### 🕸️ Wage Diversification Index (Inverse Payroll HHI)
* **Current Cohort Result:** `{w_diversification:.4f}`
* **Spectrum of Expectations:**
  * **High (≥ 0.920):** Resilient, diverse regional economy. Total corporate payroll is spread smoothly across many distinct industries, insulating the market from single-sector crashes.
  * **Average (0.750 to 0.919):** Normal industry balance. Typical industrial mix with mild leaning toward localized anchor fields.
  * **Low (< 0.750):** Monopolized / Vulnerable "Company Town." Payroll concentration is dominated by a few corporate players, leaving local debt service vulnerable if that sector slips.

### ⚡ Wage-to-Filer Disconnect Index
* **Current Cohort Result:** `{w_disconnect:+.4f}`
* **Spectrum of Expectations:**
  * **High (≥ +0.040):** Widening structural gap. IRS resident taxpayer wage growth is outpacing local business job pay, signaling affluent commuters moving in or gentrification.
  * **Average (-0.039 to +0.039):** Symmetric alignment. Wage growth inside local establishments matches local resident income tracking closely.
  * **Low (< -0.040):** Commercial operational stress. Local business wages are outstripping resident income growth, indicating rising business overhead or local worker scarcity.

---

## 🌋 DIMENSION 4: SYSTEMIC SOVEREIGN & STATE SHOCK ANCHORS (FRED / CENSUS STATE)

### 🏎️ State Coincident Momentum (YoY Activity Shift)
* **Current Cohort Result:** `{s_momentum:+.4f} ({s_momentum*100:+.2f}%)`
* **Spectrum of Expectations:**
  * **High (≥ +0.035):** High-velocity boom track. The state economy is expanding rapidly, providing strong macro tailwinds for local business revenues.
  * **Average (0.000 to +0.034):** Steady, sustainable economic growth. Solid foundation for standard long-term underwriting horizons.
  * **Low (< 0.000):** Systemic state-level recession. The regional economy is actively contracting, signaling elevated credit risks across all portfolios.

### 📊 Supplementary Sovereign Tail Risks
* **Sovereign Yield Spread (T10Y2Y):** `{y_spread:+.2f}%` *(Negative values signal an inverted curve and macro recession warning flags.)*
* **Macro Consumer Sentiment (UMCSENT):** `{c_sentiment:.1f}` *(Baseline: Historic Mean ~85.0. Scores below 65.0 indicate broad consumer caution.)*
* **State Private Labor Turnover Rate:** `{p_turnover*100:.2f}%` *(Measures quarterly workplace separations. Scores > 10% warn of high talent replacement costs.)*
* **State Net Job Flow Count:** `{n_job_flow:,}` *({year} net headcount change across all enterprise payrolls.)*

---
##### *CONFIDENTIAL INSTITUTIONAL USE ONLY — GENERATED SECURELY VIA MACROFEATUREENGINEV2 CALCULATION LAYER.*
"""
    return md


In [4]:
restaurant_metrics = engine.extract_comprehensive_macro_profile(county_fips="48085", naics_4d="7225",loan_vintage_year=2024)

generate_markdown_loan_dashboard(industry_metrics)


'# 🛸 ENTERPRISE UNDERWRITING RISK REPORT: MACRO-METRIC ENGINE\n---\n### 📅 COHORT TARGET: FIPS [48085] | VINTAGE YEAR [2024] | NAICS [7225]\n**Spatial Governance Execution Flag:** `TRUE_METRO`\n\n---\n\n## 💎 DIMENSION 1: PASSIVE WEALTH DEPTH & MIGRATION PROFILES (IRS SOI)\n\n### 📈 Macro Wealth Cushion\n* **Current Cohort Result:** `0.0494`\n* **Spectrum of Expectations:**\n  * **High (≥ 0.120):** Strong local liquidity cushion. High concentration of investment dividends and interest relative to labor wages, indicating local wealth insulation.\n  * **Average (0.040 to 0.119):** Standard balanced marketplace. Consumer spending is supported primarily by active jobs, with normal household savings reserves.\n  * **Low (< 0.040):** High-risk capital constraints. The community depends entirely on immediate payroll cycles; households lack passive liquidity cushions during stress.\n\n### 🚀 Filer Density Velocity\n* **Current Cohort Result:** `+0.1481 (+14.81%)`\n* **Spectrum of Expectations:**\n